In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from shapely.ops import substring
import numpy as np
from rasterstats import zonal_stats
from shapely.geometry import LineString, MultiLineString, Polygon


In [ ]:
mrt_path = "../../../datos-notebooks/1.HR-data-bologna/1.5.MRT-prediction/MRT_1m.tif"
utai_path = "../../../datos-notebooks/1.HR-data-bologna/1.5.MRT-prediction/UTAI_1m.tif"
walkable_paths_path = "../../../datos-notebooks/paths_bologna.geojson"
walkable_paths_utai_path = "../../../datos-notebooks/1.HR-data-bologna/1.5.MRT-prediction/walkable_paths_utai.geojson"

### index computing

In [ ]:
# Read raster
with rasterio.open(mrt_path) as src:
    mrt = src.read(1).astype(float)
    profile = src.profile
    crs = src.crs
    transform = src.transform
    nodata = src.nodata

# Create thermal adaptability index (1–10)
# 10 = lowest MRT, 1 = highest MRT

mrt_min = np.nanmin(mrt)
mrt_max = np.nanmax(mrt)

mrt_norm = (mrt - mrt_min) / (mrt_max - mrt_min) # Normalize to [0, 1]
mrt_norm_inv = 1 - mrt_norm # Invert scale (low MRT = high score)
thermal_index = 1 + 9 * mrt_norm_inv # Scale to [1, 10]
thermal_index_int = np.rint(thermal_index)   # round to nearest integer
thermal_index_int = thermal_index_int.astype(np.float32)  # keep NaN



In [ ]:
profile_out = profile.copy()
profile_out.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan
)

with rasterio.open(utai_path, "w", **profile_out) as dst:
    dst.write(thermal_index_int.astype(np.float32), 1)


In [ ]:
with rasterio.open(utai_path) as src:
    thermal_index_int = src.read(1)

plt.figure(figsize=(8, 8))
img = plt.imshow(
    thermal_index_int,
    cmap="coolwarm_r",
    vmin=1,
    vmax=10
)
plt.colorbar(img, label="Thermal Adaptability Index (1–10)")
plt.title("Urban Thermal Adaptability Index ")
plt.axis("off")
plt.show()


### reading and fixing walkable (and non walkable) paths layer

In [ ]:
gdf = gpd.read_file(walkable_paths_path)

print("CRS:", gdf.crs)
print("Geometry types:")
print(gdf.geometry.geom_type.value_counts())

In [ ]:
gdf_lines = gdf.copy()

In [ ]:
def to_lines(geom):
    if geom is None:
        return None
    if geom.geom_type in ["LineString", "MultiLineString"]:
        return geom
    if geom.geom_type == "Polygon":
        return geom.boundary
    if geom.geom_type == "MultiPolygon":
        return geom.boundary
    return None

gdf_lines["geometry"] = gdf_lines.geometry.apply(to_lines)

In [ ]:
gdf_lines = gdf_lines.to_crs(epsg=32632)  # meters
gdf_lines["length_m"] = gdf_lines.geometry.length

In [ ]:
print("Geometry types:")
print(gdf_lines.geometry.geom_type.value_counts())

In [ ]:
# Histogram of line lengths
plt.hist(gdf_lines["length_m"], bins=10)
plt.xlabel("Length (m)")
plt.ylabel("Frequency")
plt.title("line lengths")
plt.show()


In [ ]:
def split_line_max_length(line, max_len=100):
    """
    Split a LineString into segments with max length <= max_len (meters)
    """
    length = line.length
    if length <= max_len:
        return [line]

    distances = np.arange(0, length, max_len)
    segments = []

    for d0, d1 in zip(distances[:-1], distances[1:]):
        segments.append(substring(line, d0, d1))

    # last segment
    segments.append(substring(line, distances[-1], length))

    return segments

In [ ]:
rows = []

for idx, row in gdf_lines.iterrows():
    parts = split_line_max_length(row.geometry, max_len=100)

    for geom in parts:
        rows.append({
            **row.drop("geometry"),
            "geometry": geom
        })

In [ ]:
gdf_split = gpd.GeoDataFrame(rows, crs=gdf_lines.crs)
gdf_split["length_m"] = gdf_split.geometry.length


In [ ]:
# Histogram of line lengths
plt.hist(gdf_split["length_m"], bins=30)
plt.xlabel("Length (m)")
plt.ylabel("Frequency")
plt.title("line lengths")
plt.show()

In [ ]:
# Geometry type(s)
print("Geometry types:")
print(gdf_split.geometry.geom_type.value_counts())

### extract index values

In [ ]:
with rasterio.open(utai_path) as src:
    raster_crs = src.crs
    raster_transform = src.transform
    raster_nodata = src.nodata


In [ ]:
if gdf_split.crs != raster_crs:
    gdf_split = gdf_split.to_crs(raster_crs)

stats = zonal_stats(
    gdf_split,
    utai_path,
    stats=["majority"],
    nodata=raster_nodata,
    geojson_out=False
)

gdf_split["utai"] = [s["majority"] for s in stats]

In [ ]:
gdf_split.to_file(walkable_paths_utai_path, driver="GeoJSON")

In [ ]:
ax = gdf_split.plot(
    column="utai",
    cmap="viridis",
    legend=True,
    figsize=(6, 6),
    linewidth=2
)

ax.set_title("walkable paths – index")
ax.set_axis_off()
